# 1. Profile the bag to get metadata + topics

In [ ]:
import sys
from pathlib import Path

demo_dir = Path.cwd()
repo_root = demo_dir.parent if demo_dir.name == "demo" else demo_dir
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from hephaes import Profiler

bag_files = [
    str(Path("input") / "ros2.mcap"),
]
topics = []

if not bag_files:
    print("No bag files configured.")
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for t in meta.topics:
            print(f"  - {t.name} ({t.message_type}): {t.message_count} messages, {t.rate_hz} Hz")
        print("-" * 40)


# 2. Create a MappingTemplate

In [ ]:
from hephaes.mappers import build_mapping_template

mapping = build_mapping_template(topics)

# 3. Use Converter to write parquet output


In [ ]:
from pathlib import Path
from hephaes import Converter, ParquetOutputConfig, ResampleConfig, TFRecordOutputConfig

output_root = Path("./output")
parquet_output_dir = output_root / "parquet"
tfrecord_output_dir = output_root / "tfrecord"

# Default: no resampling (union timestamps, null when a column has no value).
# parquet_files = Converter(bag_files, mapping, parquet_output_dir).convert()

# Optional: regular-grid downsample or interpolate.
resample_cfg = ResampleConfig(freq_hz=10.0, method="downsample")
parquet_files = Converter(
    bag_files,
    mapping,
    parquet_output_dir,
    output=ParquetOutputConfig(),
    resample=resample_cfg,
).convert()

parquet_files


# 4. Stream the first few rows of the parquet output


In [ ]:
from hephaes import stream_wide_parquet_rows

output_parquet = parquet_files[0]
n_rows = 5

for i, row in enumerate(stream_wide_parquet_rows(output_parquet)):
    print(row)
    if i + 1 >= n_rows:
        break


# 5. Convert the same bag to TFRecord


In [ ]:
tfrecord_files = Converter(
    bag_files,
    mapping,
    tfrecord_output_dir,
    output=TFRecordOutputConfig(),
    resample=resample_cfg,
).convert()

tfrecord_files


# 6. Stream the first few rows of the TFRecord output


In [ ]:
from hephaes import stream_tfrecord_rows

output_tfrecord = tfrecord_files[0]

for i, row in enumerate(stream_tfrecord_rows(output_tfrecord)):
    print(row)
    if i + 1 >= n_rows:
        break
